# 📖 Notebook 1: Indexing Fundamentals

Indexes are the single most important tool for PostgreSQL performance. Without them, every query scans the **entire table** row by row. With the right indexes, the same query can be thousands of times faster.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why full table scans are slow and how to spot them with `EXPLAIN ANALYZE`
- How B-tree indexes work and when to use them
- How composite indexes speed up multi-column queries
- How partial indexes save space and boost performance
- How covering indexes eliminate table lookups entirely

## The Pattern: BAD → BETTER → BEST

| Approach | Technique | Speed |
|----------|-----------|-------|
| 🔴 BAD | No indexes (full table scan) | Slowest |
| 🟡 BETTER | Basic B-tree index | Faster |
| 🟢 BEST | Composite / Partial / Covering indexes | Fastest |

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 03-technologies/databases/postgres
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import time
from tabulate import tabulate

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "postgres_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    """Open a fresh connection. Use for one-off queries."""
    return psycopg2.connect(**DB_CONFIG)

def run_query(sql, params=None, fetch=True):
    """Run a query and return results."""
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql, params)
    result = cur.fetchall() if fetch else None
    cols = [desc[0] for desc in cur.description] if cur.description else []
    conn.close()
    return result, cols

def explain(sql, params=None):
    """Run EXPLAIN ANALYZE and print the query plan."""
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {sql}", params)
    plan = cur.fetchall()
    conn.close()
    print("┌─── EXPLAIN ANALYZE ───────────────────────")
    for row in plan:
        print(f"│ {row[0]}")
    print("└─────────────────────────────────────────────────────")

def timed_query(sql, params=None, label="Query", runs=5):
    """Run a query several times on ONE connection and report the average.

    We reuse a single connection so the timing reflects the query work
    itself — not the cost of opening a new TCP connection every call.
    A warm-up run is executed first and discarded so caches are primed.
    """
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    try:
        # Warm-up (not counted) — lets PostgreSQL load pages into cache
        cur.execute(sql, params)
        cur.fetchall()
        times = []
        for _ in range(runs):
            start = time.perf_counter()
            cur.execute(sql, params)
            cur.fetchall()
            times.append((time.perf_counter() - start) * 1000)
    finally:
        conn.close()
    avg = sum(times) / len(times)
    print(f"⏱️  {label}: {avg:.2f} ms (avg of {runs} runs, warm cache)")
    return avg

# Test connection
try:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM posts")
    count = cur.fetchone()[0]
    conn.close()
    print(f"✅ Connected to PostgreSQL — {count:,} posts in database")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker compose up -d")


## 🔴 BAD: No Indexes (Full Table Scan)

Let's start with the **worst case**. Imagine we want to find all posts by a specific user. Without an index, PostgreSQL has to read **every single row** in the table to find matches.

This is called a **Sequential Scan (Seq Scan)** — it's like searching for a name in a phone book by reading every page from cover to cover.

Let's see it in action:

In [ ]:
# First, let's make sure there are NO indexes on the columns we'll query
# (Primary key indexes exist, but we won't query by id)

# Drop any indexes that might exist from previous runs
drop_indexes = [
    "DROP INDEX IF EXISTS idx_posts_user_id",
    "DROP INDEX IF EXISTS idx_posts_user_created",
    "DROP INDEX IF EXISTS idx_posts_status",
    "DROP INDEX IF EXISTS idx_posts_published_user",
    "DROP INDEX IF EXISTS idx_posts_covering",
    "DROP INDEX IF EXISTS idx_comments_post_id",
]
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
for sql in drop_indexes:
    cur.execute(sql)
conn.close()
print("🧹 Cleaned up any existing indexes")
print()

# Now let's see what happens when we search WITHOUT an index
print("=" * 60)
print("🔴 BAD: Finding all posts by user_id=42 (NO INDEX)")
print("=" * 60)
print()

explain("SELECT * FROM posts WHERE user_id = 42")

In [ ]:
# Let's understand what we just saw:
# - "Seq Scan on posts" = PostgreSQL read EVERY row in the table
# - "Filter: (user_id = 42)" = it checked each row against our condition
# - "Rows Removed by Filter" = most rows didn't match (wasted work!)

# Let's also check some other common queries WITHOUT indexes:

print("🔴 BAD: Finding posts in a date range (NO INDEX)")
print()
explain("SELECT * FROM posts WHERE created_at > NOW() - INTERVAL '7 days'")
print()

print("🔴 BAD: Finding published posts by a user (NO INDEX)")
print()
explain(
    "SELECT * FROM posts WHERE user_id = 42 AND status = 'published'"
)

In [ ]:
# Let's measure the actual time for these queries

print("⏱️  Timing: Queries WITHOUT any indexes")
print("=" * 60)
print()

bad_t1 = timed_query(
    "SELECT * FROM posts WHERE user_id = 42",
    label="Posts by user_id"
)
bad_t2 = timed_query(
    "SELECT * FROM posts WHERE created_at > NOW() - INTERVAL '7 days'",
    label="Posts in last 7 days"
)
bad_t3 = timed_query(
    "SELECT * FROM posts WHERE user_id = 42 AND status = 'published'",
    label="Published posts by user"
)

print()
print("💡 All queries do Seq Scans — they read ALL 100,000 rows every time!")
print("   As the table grows to millions of rows, these queries get MUCH worse.")

## 🟡 BETTER: Basic B-tree Index

A **B-tree index** is like the index at the back of a textbook. Instead of reading every page, you look up the topic in the index and jump straight to the right page.

PostgreSQL's default index type is B-tree. It works great for:
- **Equality**: `WHERE user_id = 42`
- **Ranges**: `WHERE created_at > '2024-01-01'`
- **Sorting**: `ORDER BY created_at` (if the index column matches)

```
                    B-tree Index on user_id
                    ┌──────────────┐
                    │  Root: [500] │
                    └──┬───────┬───┘
              ┌────────┘       └────────┐
        ┌─────┴──────┐           ┌──────┴─────┐
        │ [100, 250] │           │ [750, 900] │
        └──┬────┬────┘           └──┬────┬────┘
           │    │                   │    │
    ┌──────┘    └──────┐    ┌──────┘    └──────┐
    │ Leaf: rows │     │    │ Leaf: rows │     │
    │ user_id<100│     │    │ 500<uid<750│     │
    └────────────┘     │    └────────────┘     │
                       │                       │
                  Leaf: rows              Leaf: rows
```

Instead of scanning 100,000 rows, the B-tree lets PostgreSQL find matching rows in just 3-4 lookups.

In [ ]:
# Create a basic B-tree index on user_id
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE INDEX idx_posts_user_id ON posts(user_id)")
conn.close()

print("✅ Created index: idx_posts_user_id ON posts(user_id)")
print()

# Now let's see the SAME query with the index
print("=" * 60)
print("🟡 BETTER: Finding posts by user_id=42 (WITH B-tree INDEX)")
print("=" * 60)
print()

explain("SELECT * FROM posts WHERE user_id = 42")

In [ ]:
# Look at the difference!
# - "Index Scan using idx_posts_user_id" instead of "Seq Scan"
# - Much fewer rows read — only the matching ones
# - Execution time should be dramatically lower

# Let's measure the improvement
print("⏱️  Timing: Single-column index on user_id")
print("=" * 60)
print()

better_t1 = timed_query(
    "SELECT * FROM posts WHERE user_id = 42",
    label="Posts by user_id (WITH index)"
)

print()
speedup = bad_t1 / better_t1 if better_t1 > 0 else float('inf')
print(f"🚀 Speedup: {speedup:.1f}× faster than without index!")
print()

# But what about our multi-column query?
print("🤔 Does the index help with multi-column queries?")
print()
explain(
    "SELECT * FROM posts WHERE user_id = 42 AND status = 'published'"
)
print()
print("💡 The index helps with user_id, but PostgreSQL still has to")
print("   filter by status AFTER finding the user's posts.")
print("   We can do better with a composite index...")

## 🟢 BEST: Advanced Index Strategies

Now let's look at three powerful index types that go beyond basic B-tree:

### 1. Composite Index (multi-column)
Index on `(user_id, status)` — handles both columns in one index lookup.

### 2. Partial Index
Index only the rows you actually query — e.g., only `published` posts.

### 3. Covering Index (INCLUDE)
Includes extra columns IN the index so PostgreSQL never needs to read the actual table row — called an **Index-Only Scan**.

In [ ]:
# ── COMPOSITE INDEX ──────────────────────────────────────
# When you frequently query by (user_id AND status), a composite
# index handles BOTH conditions in one index lookup.

conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE INDEX idx_posts_user_status ON posts(user_id, status)")
conn.close()

print("✅ Created composite index: (user_id, status)")
print()

print("=" * 60)
print("🟢 BEST #1: Composite index for multi-column queries")
print("=" * 60)
print()

explain(
    "SELECT * FROM posts WHERE user_id = 42 AND status = 'published'"
)
print()

best_t3 = timed_query(
    "SELECT * FROM posts WHERE user_id = 42 AND status = 'published'",
    label="Published posts by user (composite index)"
)
print()
speedup = bad_t3 / best_t3 if best_t3 > 0 else float('inf')
print(f"🚀 Speedup vs no index: {speedup:.1f}×")

In [ ]:
# ── PARTIAL INDEX ────────────────────────────────────────
# If 80% of your queries are for 'published' posts, why index
# draft and archived posts? A partial index saves space and is faster.

conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("""
    CREATE INDEX idx_posts_published_user
    ON posts(user_id, created_at)
    WHERE status = 'published'
""")
conn.close()

print("✅ Created partial index: (user_id, created_at) WHERE status = 'published'")
print()

print("=" * 60)
print("🟢 BEST #2: Partial index — only indexes published posts")
print("=" * 60)
print()

explain(
    "SELECT * FROM posts WHERE user_id = 42 AND status = 'published' ORDER BY created_at DESC LIMIT 10"
)
print()

# Let's compare index sizes
results, cols = run_query("""
    SELECT indexname, pg_size_pretty(pg_relation_size(indexname::regclass)) AS size
    FROM pg_indexes
    WHERE tablename = 'posts' AND indexname IN (
        'idx_posts_user_id', 'idx_posts_user_status', 'idx_posts_published_user'
    )
    ORDER BY pg_relation_size(indexname::regclass) DESC
""")

print("📏 Index Size Comparison:")
print(tabulate(results, headers=cols, tablefmt="simple_grid"))
print()
print("💡 The partial index is smaller because it only indexes ~60% of rows!")

In [ ]:
# ── COVERING INDEX (INCLUDE) ─────────────────────────────
# A covering index includes extra columns IN the index itself.
# This means PostgreSQL can answer the query entirely from the
# index without ever reading the actual table row.
# This is called an "Index-Only Scan" — the fastest possible read.

conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("""
    CREATE INDEX idx_posts_covering
    ON posts(user_id, created_at DESC)
    INCLUDE (title, like_count)
    WHERE status = 'published'
""")
conn.close()

print("✅ Created covering index: (user_id, created_at) INCLUDE (title, like_count)")
print()

print("=" * 60)
print("🟢 BEST #3: Covering index — Index-Only Scan, no table access")
print("=" * 60)
print()

# This query can be answered entirely from the index!
explain(
    "SELECT title, like_count, created_at FROM posts WHERE user_id = 42 AND status = 'published' ORDER BY created_at DESC LIMIT 10"
)
print()

best_covering = timed_query(
    "SELECT title, like_count, created_at FROM posts WHERE user_id = 42 AND status = 'published' ORDER BY created_at DESC LIMIT 10",
    label="Covering index (Index-Only Scan)"
)
print()
print("💡 Notice 'Index Only Scan' in the plan — PostgreSQL never touches the table!")
print("   This is the fastest possible way to answer this query.")

## 📊 Final Comparison

Let's put it all together and see how each approach compares.

In [ ]:
# Re-run all approaches for a fair comparison

# 1. Drop all indexes, measure BAD
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
for idx in ['idx_posts_user_id', 'idx_posts_user_status',
            'idx_posts_published_user', 'idx_posts_covering']:
    cur.execute(f"DROP INDEX IF EXISTS {idx}")
conn.close()

query = "SELECT title, like_count, created_at FROM posts WHERE user_id = 42 AND status = 'published' ORDER BY created_at DESC LIMIT 10"

bad = timed_query(query, label="🔴 BAD: No index (Seq Scan)")

# 2. Basic index
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE INDEX idx_posts_user_id ON posts(user_id)")
conn.close()
better = timed_query(query, label="🟡 BETTER: Basic B-tree (user_id)")

# 3. Composite + partial + covering
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("""
    CREATE INDEX idx_posts_covering
    ON posts(user_id, created_at DESC)
    INCLUDE (title, like_count)
    WHERE status = 'published'
""")
conn.close()
best = timed_query(query, label="🟢 BEST: Covering index")

print()
print("=" * 60)
print("📊 SUMMARY")
print("=" * 60)
print()
print(f"  🔴 No index:       {bad:>8.2f} ms")
print(f"  🟡 Basic B-tree:   {better:>8.2f} ms  ({bad/better:.0f}× faster)")
print(f"  🟢 Covering index: {best:>8.2f} ms  ({bad/best:.0f}× faster)")
print()
print("💡 Key Takeaway: The right index strategy can make queries")
print("   100× to 1000× faster. But each index has a cost:")
print("   - More disk space")
print("   - Slower writes (inserts/updates must update the index)")
print("   - Index the columns you QUERY, not every column!")

## ⚠️ Common Index Mistakes

| Mistake | Why It's Bad |
|---------|-------------|
| Indexing every column | Wastes disk, slows writes |
| Wrong column order in composite index | `(status, user_id)` won't help `WHERE user_id = 42` |
| Not using partial indexes | Indexing rows you never query |
| Forgetting to ANALYZE | Planner uses stale statistics |
| Too many overlapping indexes | Redundant indexes waste resources |

## 🧹 Cleanup

Run this cell to remove indexes created during the lab (the next notebook starts fresh).

## 🌟 Bonus: BRIN Indexes for Huge Time-Series Tables

A **BRIN** (Block Range INdex) is a tiny index that only stores the *min/max* value for each block of pages. It's useless for random lookups but perfect for **append-only, time-ordered tables** (logs, metrics, events) where rows with similar `created_at` are physically close on disk.

- **Size**: a BRIN index on 100M rows may be only a few MB (a B-tree would be gigabytes).
- **Good for**: `WHERE created_at BETWEEN ... AND ...` on data inserted in chronological order.
- **Bad for**: random/UUID columns, or tables where recent rows are mixed with old ones.


In [ ]:
# Compare a B-tree index and a BRIN index on posts.created_at
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("DROP INDEX IF EXISTS idx_posts_created_btree")
cur.execute("DROP INDEX IF EXISTS idx_posts_created_brin")
cur.execute("CREATE INDEX idx_posts_created_btree ON posts(created_at)")
cur.execute("CREATE INDEX idx_posts_created_brin  ON posts USING BRIN (created_at)")
conn.close()

results, cols = run_query("""
    SELECT indexname,
           pg_size_pretty(pg_relation_size(indexname::regclass)) AS size
    FROM pg_indexes
    WHERE indexname IN ('idx_posts_created_btree', 'idx_posts_created_brin')
    ORDER BY pg_relation_size(indexname::regclass) DESC
""")
print("📏 B-tree vs BRIN on posts.created_at:")
print(tabulate(results, headers=cols, tablefmt='simple_grid'))
print()
print("💡 The BRIN index is dramatically smaller. On a 100M-row logs table")
print("   the difference is often GB vs a few MB — and range scans are still")
print("   fast because rows are physically clustered by insertion time.")

# Clean up so the final comparison isn't biased
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("DROP INDEX IF EXISTS idx_posts_created_btree")
cur.execute("DROP INDEX IF EXISTS idx_posts_created_brin")
conn.close()


In [ ]:
# Clean up indexes created in this notebook
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
for idx in ['idx_posts_user_id', 'idx_posts_user_status',
            'idx_posts_published_user', 'idx_posts_covering',
            'idx_comments_post_id']:
    cur.execute(f"DROP INDEX IF EXISTS {idx}")
conn.close()
print("🧹 All custom indexes removed")

## 📚 Summary

### Key Takeaways

1. **Without indexes**, PostgreSQL does a Sequential Scan — reads every row
2. **B-tree indexes** are the default and handle equality, range, and sort queries
3. **Composite indexes** handle multi-column `WHERE` clauses efficiently
4. **Partial indexes** save space by only indexing rows you actually query
5. **Covering indexes** (INCLUDE) enable Index-Only Scans — the fastest reads possible
6. **Every index has a cost** — more disk, slower writes. Index strategically!

### Next Up

In **Notebook 2**, we'll optimize the **queries themselves** — because even with perfect indexes, a badly written query can still be slow.